# Tazama Data Lakehouse — Catalog & Browser

Runs against the **live** Hudi warehouse at `WAREHOUSE_ROOT` (default `/opt/Tazama_Warehouse`).

**What this notebook does:**
1. Starts a minimal Spark session (read-only, no ETL)
2. Auto-discovers every Hudi table by scanning for `.hoodie/` marker directories
3. Prints a summary catalog: path · row count · record key · latest commit time
4. Lets you deep-dive any table: full schema, sample rows, commit history, partition map

Run cells top-to-bottom. Each section after §3 is independent — skip what you don't need.

In [ ]:

TENANT_FILTER_VALUE = "TAZAMA"
TENANT_FILTER_COLUMNS = ("tenant_id", "tx_tenant_id", "tenantid")

# Load Hudi tables through the dashboard tenant scope.
def load_tenant_hudi(path, **options):
    reader = spark.read.format("hudi")
    for key, value in options.items():
        reader = reader.option(key, value)
    df = reader.load(path)
    tenant_col = next((c for c in TENANT_FILTER_COLUMNS if c in df.columns), None)
    if tenant_col is None:
        raise ValueError(
            f"Hudi table at {path} has no tenant filter column; "
            f"expected one of {TENANT_FILTER_COLUMNS}"
        )
    return df.filter(df[tenant_col] == TENANT_FILTER_VALUE)

print(f"Tenant filter active: {TENANT_FILTER_VALUE}")


In [25]:
# §1 — Spark session (read-only, Hudi + S3A/Ozone support)
import os
import pathlib
from pyspark.sql import SparkSession

spark_jars   = os.environ.get("SPARK_JARS", "")
jar_list     = spark_jars.split(",") if spark_jars else []
s3a_endpoint = os.environ.get("S3A_ENDPOINT", "")
s3a_access   = os.environ.get("S3A_ACCESS_KEY", "")
s3a_secret   = os.environ.get("S3A_SECRET_KEY", "")

builder = (
    SparkSession.builder
    .appName("tazama-lakehouse-catalog")
    .master("local[2]")          # 2 cores — enough for reads
    .config("spark.jars", spark_jars)
    .config("spark.driver.extraClassPath", ":".join(jar_list))
    .config("spark.executor.extraClassPath", ":".join(jar_list))
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.spark.HoodieSparkKryoRegistrar")
    .config("spark.sql.extensions",
            "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    .config("spark.driver.memory",
            os.environ.get("SPARK_DRIVER_MEMORY", "2g"))
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
)

if s3a_endpoint:
    builder = (
        builder
        .config("spark.hadoop.fs.s3a.endpoint", s3a_endpoint)
        .config("spark.hadoop.fs.s3a.access.key", s3a_access)
        .config("spark.hadoop.fs.s3a.secret.key", s3a_secret)
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.hadoop.fs.s3a.impl",
                "org.apache.hadoop.fs.s3a.S3AFileSystem")
    )

try:
    spark.stop()
except NameError:
    pass

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "ready.")

In [30]:
# §2 — Discover all Hudi tables under WAREHOUSE_ROOT
#
# A Hudi table is identified by a .hoodie/ sub-directory.
# We walk the tree and collect every directory that contains one.

WAREHOUSE_ROOT = os.environ.get("WAREHOUSE_ROOT", "/opt/Tazama_Warehouse")
print(f"Scanning: {WAREHOUSE_ROOT}")

hudi_tables = []   # list of (layer, name, abs_path)

for root, dirs, files in os.walk(WAREHOUSE_ROOT):
    if ".hoodie" in dirs:
        rel   = os.path.relpath(root, WAREHOUSE_ROOT)   # e.g. gold/transactions
        parts = pathlib.PurePosixPath(rel.replace("\\", "/")).parts
        layer = parts[0] if len(parts) >= 2 else "(root)"
        name  = "/".join(parts[1:]) if len(parts) >= 2 else parts[0]
        hudi_tables.append((layer, name, root))
        dirs[:] = []   # don't recurse into Hudi table directories

hudi_tables.sort()
print(f"Found {len(hudi_tables)} Hudi table(s):\n")
for layer, name, path in hudi_tables:
    print(f"  [{layer:8s}]  {name}")
    print(f"             {path}")

In [31]:
# §3 — Summary catalog: row count + latest commit per table
#
# Reads .hoodie/hoodie.properties (instant, no Spark) for table name / record key,
# then uses Spark only for the row count so startup cost is paid once.

import time

# Define the read_hoodie_props helper used by later cells.
def read_hoodie_props(table_path):
    """Parse .hoodie/hoodie.properties without Spark."""
    props_file = os.path.join(table_path, ".hoodie", "hoodie.properties")
    props = {}
    if os.path.isfile(props_file):
        with open(props_file) as f:
            for line in f:
                line = line.strip()
                if "=" in line and not line.startswith("#"):
                    k, _, v = line.partition("=")
                    props[k.strip()] = v.strip()
    return props

# Define the latest_commit helper used by later cells.
def latest_commit(table_path):
    """Return the most recent completed commit timestamp string, or None."""
    timeline_dir = os.path.join(table_path, ".hoodie")
    commits = sorted(
        f for f in os.listdir(timeline_dir)
        if f.endswith(".commit") or f.endswith(".deltacommit")
    )
    if commits:
        ts = commits[-1].split(".")[0]   # yyyyMMddHHmmss
        return f"{ts[0:4]}-{ts[4:6]}-{ts[6:8]} {ts[8:10]}:{ts[10:12]}:{ts[12:14]}"
    return "(no commits)"

rows_data = []
for layer, name, path in hudi_tables:
    props      = read_hoodie_props(path)
    tbl_name   = props.get("hoodie.table.name", "?")
    record_key = props.get("hoodie.datasource.write.recordkey.field", "?")
    tbl_type   = props.get("hoodie.table.type", "?")
    last_commit = latest_commit(path)

    t0  = time.time()
    # Load the authoritative Hudi table used by downstream metrics.
    cnt = load_tenant_hudi(path).count()
    elapsed = time.time() - t0

    rows_data.append({
        "layer":      layer,
        "table":      name,
        "hudi_name":  tbl_name,
        "type":       tbl_type,
        "record_key": record_key,
        "rows":       cnt,
        "last_commit": last_commit,
        "path":       path,
    })
    print(f"  {layer}/{name:30s}  {cnt:>8,} rows   last commit: {last_commit}")

print("\nCatalog complete.")

In [ ]:
# §3b — Render catalog as a pandas DataFrame for a cleaner view
import pandas as pd

catalog_df = pd.DataFrame(rows_data)[["layer", "table", "type", "record_key", "rows", "last_commit"]]
catalog_df.sort_values(["layer", "table"]).reset_index(drop=True)

---
## Deep-dive a specific table

Set `TARGET` to any `layer/name` value from the catalog above, then run §4 onwards.

In [76]:
# §4 — Pick a table to inspect
TARGET = "gold/transactions"    # ← change me

table_meta = next((r for r in rows_data if f"{r['layer']}/{r['table']}" == TARGET), None)
if table_meta is None:
    raise ValueError(f"{TARGET!r} not found in catalog. Available:\n"
                     + "\n".join(f"  {r['layer']}/{r['table']}" for r in rows_data))

# Load the authoritative Hudi table used by downstream metrics.
df = load_tenant_hudi(table_meta["path"])
print(f"Loaded  : {TARGET}")
print(f"Path    : {table_meta['path']}")
print(f"Rows    : {table_meta['rows']:,}")
print(f"Columns : {len(df.columns)}")

In [77]:
# §5 — Schema (hide Hudi internal columns by default)
SHOW_HUDI_INTERNALS = False   # set True to include _hoodie_* columns

if SHOW_HUDI_INTERNALS:
    df.printSchema()
else:
    from pyspark.sql.types import StructType
    visible = StructType([f for f in df.schema.fields
                          if not f.name.startswith("_hoodie_")])
    # Pretty-print manually so it looks the same as printSchema()
    def _print_schema(schema, indent=0):
        for field in schema.fields:
            nullable = "nullable" if field.nullable else "not nullable"
            dtype = field.dataType
            if hasattr(dtype, "fields"):   # StructType
                print(" " * indent + f"|-- {field.name}: struct ({nullable})")
                _print_schema(dtype, indent + 4)
            elif hasattr(dtype, "elementType"):   # ArrayType
                print(" " * indent + f"|-- {field.name}: array ({nullable})")
            else:
                print(" " * indent + f"|-- {field.name}: {dtype.simpleString()} ({nullable})")
    print(f"root ({len(visible.fields)} business columns, _hoodie_* hidden)")
    _print_schema(visible)

In [78]:
# §6 — Sample rows (most recent 10 by event time)
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType, DateType

# Define the to_pandas_safe helper used by later cells.
def to_pandas_safe(sdf):
    """Cast timestamp/date cols to string to avoid pandas 2.x tz conversion error."""
    for field in sdf.schema.fields:
        if isinstance(field.dataType, (TimestampType, DateType)):
            sdf = sdf.withColumn(field.name, F.col(field.name).cast("string"))
    return sdf.toPandas()

# Prefer a business event timestamp over the Hudi internal commit time.
# This ensures the most recently TMS-processed records (with DataCache populated)
# appear first rather than the most recently ingested Hudi commits.
_EVENT_TS_CANDIDATES = ["credttm", "cre_dt_tm", "dc_cre_dt_tm", "event_ts", "created_at", "tx_dt"]
_sort_col = next(
    (c for c in _EVENT_TS_CANDIDATES if c in df.columns),
    "_hoodie_commit_time"   # fallback: Hudi internal ingestion timestamp
)

business_cols = [c for c in df.columns if not c.startswith("_hoodie_")]
sample = (
    df.select("_hoodie_commit_time", *business_cols)
      .orderBy(_sort_col, ascending=False)
      .limit(10)
)
print(f"Sorted by: {_sort_col} (descending)")
to_pandas_safe(sample)


In [79]:
# §6b — Single record detail (all fields, fully expanded)
#
# Set MESSAGE_ID and TENANT_ID to pin a specific record.
# Leave both empty to show the most-recent record (by the same sort column as §6).
#
# The DataFrame is transposed so each field is on its own row — making
# long values (JSON payloads, nested structs, DataCache blocks) easy to read.

MESSAGE_ID = ""   # e.g. "0db3c737447347b0bf38dc466e0743c6"
TENANT_ID  = ""   # e.g. "PAYSYSLABS"

# Determine the record key field name from Hudi properties (fallback: common names)
_props = read_hoodie_props(table_meta["path"])
_rk    = _props.get("hoodie.datasource.write.recordkey.field", "")
_ID_CANDIDATES = [c for c in ([_rk] + ["msg_id", "message_id", "transaction_id", "id"])
                  if c in df.columns]
_id_col = _ID_CANDIDATES[0] if _ID_CANDIDATES else None

if MESSAGE_ID and _id_col:
    filtered = df.filter(F.col(_id_col) == MESSAGE_ID)
    if TENANT_ID:
        _TENANT_CANDIDATES = [c for c in ["tenant_id", "tx_tenant_id", "tenantid"] if c in df.columns]
        if _TENANT_CANDIDATES:
            filtered = filtered.filter(F.col(_TENANT_CANDIDATES[0]) == TENANT_ID)
    row_df = filtered.limit(1)
    print(f"Filtered by: {_id_col} = {MESSAGE_ID!r}"
          + (f"  +  tenant = {TENANT_ID!r}" if TENANT_ID else ""))
else:
    # Fall back to most-recent record using the same sort column as §6
    row_df = df.orderBy(_sort_col, ascending=False).limit(1)
    print(f"Showing most-recent record (sorted by: {_sort_col})")
    print("  → Set MESSAGE_ID above to pin a specific record.")

single_row = to_pandas_safe(row_df).T.rename(columns={0: "value"})

with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(single_row)


In [ ]:
# §7 — Commit history (how many rows landed in each commit)
(
    df.groupBy(F.substring("_hoodie_commit_time", 1, 12).alias("commit_minute"))
      .count()
      .orderBy("commit_minute", ascending=False)
      .limit(30)
      .toPandas()
)

In [ ]:
# §8 — Column-level statistics (nulls, distinct values, numeric ranges)
#
# Runs spark.sql describe() on non-struct, non-array columns.

from pyspark.sql.types import StructType, ArrayType, MapType

scalar_cols = [
    f.name for f in df.schema.fields
    if not f.name.startswith("_hoodie_")
    and not isinstance(f.dataType, (StructType, ArrayType, MapType))
]

stats = df.select(*scalar_cols).describe()
to_pandas_safe(stats)

In [58]:
# §9 — Partition map
#
# Reads _hoodie_partition_path (written on every Hudi row) instead of
# hoodie.properties, so the result is accurate even when NonpartitionedKeyGenerator
# was used — including cases where a partition field was configured but the
# generator class overrides it at write time.

from IPython.display import display

partition_counts = (
    df.groupBy("_hoodie_partition_path")
      .count()
      .orderBy("count", ascending=False)
      .limit(50)
      .toPandas()
)

unique_paths = set(partition_counts["_hoodie_partition_path"].tolist())

if unique_paths <= {"", "default"}:
    # Single empty/placeholder path — table is not partitioned
    props = read_hoodie_props(table_meta["path"])
    kg    = props.get("hoodie.datasource.write.keygenerator.class", "").split(".")[-1]
    pf    = props.get("hoodie.datasource.write.partitionpath.field", "(none)")
    print("Table is not partitioned.")
    print(f"  KeyGenerator             : {kg or '(not set)'}")
    print(f"  Partition field in props : {pf}")
    print(f"  _hoodie_partition_path   : {repr(next(iter(unique_paths), ''))}")
else:
    props = read_hoodie_props(table_meta["path"])
    pf    = props.get("hoodie.datasource.write.partitionpath.field", "(not set in properties)")
    print(f"Partition field (from properties) : {pf}")
    print(f"Distinct partition paths          : {len(unique_paths)}")
    display(partition_counts)


In [59]:
# §10 — Register all tables as Spark SQL temp views
#
# After this cell you can run arbitrary SQL in the next cell.
# View names: <layer>_<table>  (slashes → underscores)

registered = []
for r in rows_data:
    view_name = f"{r['layer']}_{r['table'].replace('/', '_')}"
    load_tenant_hudi(r["path"]).createOrReplaceTempView(view_name)
    registered.append(view_name)

print("Registered temp views:")
for v in registered:
    print(f"  {v}")

In [60]:
# §11 — Spark SQL playground
#
# Change the query below. Use the view names printed in §10.

SQL = """
SELECT
    SUBSTRING(_hoodie_commit_time, 1, 8) AS commit_day,
    COUNT(*)                             AS row_count
FROM gold_transactions
GROUP BY 1
ORDER BY 1 DESC
LIMIT 30
"""

to_pandas_safe(spark.sql(SQL))

In [ ]:
# §11b — Cross-table join: transactions → debtor/creditor accounts
#
# `dc_cdtr_acct_id` and `dc_dbtr_acct_id` live in `gold/pacs002`, NOT in
# `gold/transactions`.  They come from the Tazama TMS `DataCache` enrichment
# block embedded in the stored pacs.002 message — not from standard ISO 20022
# fields.  The full lookup chain is:
#
#   gold_transactions  ─(end_to_end_id + tenant_id)─▶  gold_pacs002
#   gold_pacs002       ─(dc_cdtr_acct_id)────────────▶  gold_account   (creditor account detail)
#   gold_pacs002       ─(dc_dbtr_acct_id)────────────▶  gold_account   (debtor account detail)
#   gold_pacs002       ─(dc_cdtr_id)──────────────────▶ gold_account_holder.counterparty_id
#   gold_pacs002       ─(dc_dbtr_id)──────────────────▶ gold_account_holder.counterparty_id
#
# `gold_transactions` stores raw ISO field names (endtoendid, tenantid, txtp,
# amt, ccy, txsts) — unlike `gold_pacs002` whose ETL renames them to
# snake_case.  Alias them in the SELECT list and use the raw names in the
# join predicate.
#
# Requires §10 (temp views) to have been run first.

SQL_ACCT_LOOKUP = """
SELECT
    t.transaction_id,
    t.endtoendid                        AS end_to_end_id,
    t.tenantid                          AS tenant_id,
    t.txtp                              AS tx_type,
    t.amt                               AS tx_amount,
    t.ccy                               AS tx_ccy,
    t.txsts                             AS tx_status,
    t.event_ts,

    -- Creditor (payee) account fields from DataCache
    p.dc_cdtr_acct_id                   AS cdtr_account_id,
    p.dc_cdtr_id                        AS cdtr_party_id,
    cdtr_acct.account_id                AS cdtr_acct_confirmed,

    -- Debtor (payer) account fields from DataCache
    p.dc_dbtr_acct_id                   AS dbtr_account_id,
    p.dc_dbtr_id                        AS dbtr_party_id,
    dbtr_acct.account_id                AS dbtr_acct_confirmed,

    -- DataCache amount (pre-resolved by TMS; may differ from pacs.008 InstructedAmount)
    p.dc_instd_amt                      AS dc_amount,
    p.dc_instd_ccy                      AS dc_ccy

FROM       gold_transactions            AS t

-- Link to pacs002 via end-to-end ID + tenant (DataCache fields only exist on gold_pacs002)
INNER JOIN gold_pacs002                 AS p
        ON  p.orgnl_end_to_end_id = t.endtoendid
        AND p.tx_tenant_id        = t.tenantid

-- Resolve creditor account from gold/account
LEFT  JOIN gold_account                 AS cdtr_acct
        ON  cdtr_acct.account_id = p.dc_cdtr_acct_id
        AND cdtr_acct.tenant_id  = t.tenantid

-- Resolve debtor account from gold/account
LEFT  JOIN gold_account                 AS dbtr_acct
        ON  dbtr_acct.account_id = p.dc_dbtr_acct_id
        AND dbtr_acct.tenant_id  = t.tenantid

ORDER BY t.event_ts DESC
LIMIT 20
"""

to_pandas_safe(spark.sql(SQL_ACCT_LOOKUP))


In [ ]:
# §12 — Hudi timeline (raw .commit files) for the target table
#
# Shows every action recorded in the Hudi timeline directory.
#
# Hudi 0.14+ uses 17-digit timestamps (yyyyMMddHHmmssSSS).
# Earlier versions used 14-digit timestamps (yyyyMMddHHmmss).
# The regex matches both.

import re
from IPython.display import display

timeline_dir = os.path.join(table_meta["path"], ".hoodie")
timeline_files = sorted(
    f for f in os.listdir(timeline_dir)
    if re.match(r"^\d{14,}\.", f)
)

if not timeline_files:
    print(f"No timeline files found in {timeline_dir}")
    print("Directory contents:")
    for entry in sorted(os.listdir(timeline_dir)):
        print(f"  {entry}")
else:
    rows_tl = []
    for f in timeline_files:
        ts_raw, _, rest = f.partition(".")
        # Normalise action label: strip trailing qualifiers into readable tags
        action = (
            rest
            .replace(".requested", " (requested)")
            .replace(".inflight",  " (inflight)")
        )
        ts = ts_raw
        ts_fmt = (
            f"{ts[0:4]}-{ts[4:6]}-{ts[6:8]} {ts[8:10]}:{ts[10:12]}:{ts[12:14]}"
            + (f".{ts[14:]}" if len(ts) > 14 else "")
            if len(ts) >= 14 else ts
        )
        size_kb = os.path.getsize(os.path.join(timeline_dir, f)) / 1024
        rows_tl.append({
            "timestamp": ts_fmt,
            "action":    action,
            "file_KB":   round(size_kb, 1),
        })

    display(pd.DataFrame(rows_tl).tail(30))


In [ ]:
# §13 — Time-travel: read the table as of a specific commit
#
# Set AS_OF_COMMIT to a timestamp from the range shown when you leave it empty.
# Hudi 0.14+ uses 17-digit timestamps (yyyyMMddHHmmssSSS).
# A 14-digit prefix is also accepted and will match the first qualifying instant.
# Leave empty to see the valid range without reading any data.

AS_OF_COMMIT = ""   # e.g. "20250101120000123"

# ── Resolve completed commits from the .hoodie timeline ──────────────────────
timeline_dir = os.path.join(table_meta["path"], ".hoodie")
completed = sorted(
    f.split(".")[0]
    for f in os.listdir(timeline_dir)
    if re.match(r"^\d{14,}\.commit$", f)   # only fully completed commits
)

# Define the fmt_ts helper used by later cells.
def fmt_ts(ts):
    """Pretty-print a raw Hudi timestamp string."""
    return (
        f"{ts[0:4]}-{ts[4:6]}-{ts[6:8]} {ts[8:10]}:{ts[10:12]}:{ts[12:14]}"
        + (f".{ts[14:]}" if len(ts) > 14 else "")
        if len(ts) >= 14 else ts
    )

if not completed:
    print("No completed commits found in the timeline — the table may still be empty.")

elif not AS_OF_COMMIT:
    # ── Guidance mode: show bounds and a sample of available instants ────────
    print("AS_OF_COMMIT is not set.  Set it to any value in the range below.\n")
    print(f"  Earliest commit : {completed[0]}")
    print(f"                    ({fmt_ts(completed[0])})")
    print(f"  Latest commit   : {completed[-1]}")
    print(f"                    ({fmt_ts(completed[-1])})")
    print(f"\n  Total completed commits : {len(completed)}")
    print("\n  Last 5 commits (most useful for time-travel):")
    for c in completed[-5:]:
        print(f"    {c}  ({fmt_ts(c)})")

else:
    # ── Execution mode: validate then read ───────────────────────────────────
    matched = [c for c in completed if c.startswith(AS_OF_COMMIT)]

    if not matched:
        print(f"ERROR: No completed commit found matching {AS_OF_COMMIT!r}\n")
        print(f"  Valid range:  {completed[0]}  →  {completed[-1]}")
        print(f"  ({fmt_ts(completed[0])}  →  {fmt_ts(completed[-1])})")
        print(f"\n  Total completed commits : {len(completed)}")
        print("\n  Last 5 commits:")
        for c in completed[-5:]:
            print(f"    {c}  ({fmt_ts(c)})")
    else:
        instant = matched[0]
        print(f"Reading snapshot at : {instant}  ({fmt_ts(instant)})")
        print(f"Valid range         : {completed[0]}  →  {completed[-1]}\n")
        df_snapshot = load_tenant_hudi(
            table_meta["path"],
            **{"as.of.instant": instant}
        )
        print(f"Rows at {instant}: {df_snapshot.count():,}")
        display(to_pandas_safe(df_snapshot.limit(5)))


In [ ]:
# §14 — Tear down Spark session when done
spark.stop()
print("Spark stopped.")